# ESM-2 as a Surrogate Model for Protein Active Learning

This tutorial introduces the ESM-2 protein language model as a frozen encoder and LoRA-finetuned surrogate for active learning. We use the GFP fluorescence dataset as the primary vehicle, with a ProteinGym DMS assay as a real-world extension.

**Prerequisites:** Complete these tutorials first:
- `tutorials/experiments/offline_design_tutorial.ipynb`
- `tutorials/models/gp_tutorial.ipynb`

**What you will learn:**
1. How ESM-2 embeddings encode protein function without task-specific training
2. How to train a regression head on frozen vs LoRA-adapted ESM-2
3. How MC Dropout provides calibrated uncertainty estimates
4. How to run a full active learning cycle with ESM-2 + UCB acquisition
5. How to transfer ESM-2 to a DMS fitness landscape (ProteinGym)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import scipy.stats
from sklearn.decomposition import PCA
from IPython.display import Image, display
from pathlib import Path

from alf_core import (
    BaseDatasetConfig,
    DatasetSearch,
    DesignTask,
    Optimizer,
    Oracle,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_tools.datasets import GFP
from alf_tools.models.esm2 import (
    ESM2DropoutModel,
    ESM2ModelConfig,
    ESM2TrainConfig,
    ESM2RegressionHead,
)
from alf_tools.models.gp import GPModel, GPModelConfig, FeaturizerConfig
from alf_tools.models.utils import extract_sequences_from_inputs
from alf_tools.optimizer.acquisition_functions import UCB

# Device detection: GPU if available, CPU fallback
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️  Running on CPU — long-running cells will be slower. "
          "See timing notes in each section.")

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED);
print("Setup complete.")

---
## Section 1: ESM-2 as a Sequence Encoder

**Goals:**
- Load the GFP dataset and inspect sequence/label distributions
- Compute ESM-2 mean-pooled embeddings for a subset of sequences
- Visualise the embedding space with PCA, coloured by fitness
- Understand why ESM-2 embeddings are valuable compared to raw sequence features

ESM-2 was pretrained on ~250M protein sequences from UniRef50. Even before any task-specific training, its 640-dimensional embeddings capture evolutionary relationships and functional properties of proteins.

In [ ]:
# GFP dataset split (used in Sections 1 and 2): 480 train, 120 val, 200 test, 200 pool
gfp_supervised = GFP(BaseDatasetConfig(
    name="gfp",
    modality="sequence",
    seed=SEED,
    train_ratio=0.6,       # 600 sequences for train+val
    validation_frac=0.2,   # 20% of train+val → 120 val, 480 train
    test_ratio=0.2,        # 200 test sequences
    split_type="random",
))
gfp_supervised.setup()

train_data = gfp_supervised.train_dataset
val_data   = gfp_supervised.validation_dataset
test_data  = gfp_supervised.test_dataset

print(gfp_supervised)

In [ ]:
# Verify expected split sizes
assert len(train_data) == 480, f"Expected 480 train, got {len(train_data)}"
assert len(val_data)   == 120, f"Expected 120 val, got {len(val_data)}"
assert len(test_data)  == 200, f"Expected 200 test, got {len(test_data)}"

# Labels are continuous brightness values
labels_all = np.array(train_data.labels)
assert labels_all.ndim == 1
print(f"Train label range: [{labels_all.min():.3f}, {labels_all.max():.3f}]")
print("Sample sequences (first 3):")
for cand in list(train_data.candidates)[:3]:
    print(f"  {cand.data[:30]}...")
print("✓ GFP dataset loaded and splits verified")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label histogram
for split, data, colour in [
    ("train", train_data, "steelblue"),
    ("val",   val_data,   "orange"),
    ("test",  test_data,  "green"),
]:
    axes[0].hist(np.array(data.labels), bins=30, alpha=0.6, label=split, color=colour)
axes[0].set_xlabel("Median Brightness"); axes[0].set_ylabel("Count")
axes[0].set_title("GFP Brightness Distribution by Split")
axes[0].legend()

# Sequence length distribution
lengths = [len(c.data) for c in list(train_data.candidates)]
axes[1].hist(lengths, bins=20, color="steelblue", alpha=0.8)
axes[1].set_xlabel("Sequence Length"); axes[1].set_ylabel("Count")
axes[1].set_title("GFP Sequence Length Distribution (Train)")

plt.tight_layout(); plt.show()

### ESM-2 Embeddings

We load a frozen ESM-2 150M encoder and compute mean-pooled sequence embeddings for 200 GFP sequences. No training happens here — these are the raw pretrained representations.

In [ ]:
%%time
# Initialise frozen model (no training yet)
frozen_model = ESM2DropoutModel(
    model_config=ESM2ModelConfig(),     # 150M, embedding_dim=640
    train_config=ESM2TrainConfig(),
    device=device,
)

# Embed 200 sequences from the training set for visualisation
N_VIZ = 200
viz_candidates = list(train_data.candidates)[:N_VIZ]
viz_labels     = np.array(train_data.labels)[:N_VIZ]

with torch.no_grad():
    embeddings = frozen_model._get_embeddings(
        [c.data for c in viz_candidates]
    ).numpy()  # (200, 640)

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
assert embeddings.shape == (N_VIZ, 640), \
    f"Expected ({N_VIZ}, 640), got {embeddings.shape}"
assert not np.any(np.isnan(embeddings)), "NaN in embeddings"
print("✓ Embeddings shape and validity verified")

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(embeddings)  # (200, 2)
explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    pca_coords[:, 0], pca_coords[:, 1],
    c=viz_labels, cmap="plasma", alpha=0.85, s=40, edgecolors="none"
)
plt.colorbar(sc, ax=ax, label="Brightness")
ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% variance)")
ax.set_title("ESM-2 Embedding Space (PCA) — Coloured by GFP Brightness")

plt.tight_layout(); plt.show()
print("Expected: bright sequences cluster together, "
      "showing ESM-2 captures functional relationships before any training.")

### Key Takeaway — Section 1

ESM-2 embeddings organise sequences by function *without any task-specific training*. Bright sequences cluster together in PCA space, showing that evolutionary pretraining captures GFP-relevant properties.

**Contrast with GP:** A Gaussian Process with one-hot sequence features treats each sequence position independently and has no concept of evolutionary context. ESM-2 provides a biologically informed starting point that makes downstream learning much more data-efficient.

| Feature | GP (one-hot) | ESM-2 (frozen) |
|---|---|---|
| Captures evolutionary context | ✗ | ✓ |
| Uncertainty estimate | Closed-form posterior | MC Dropout |
| Training data required | Any amount | Pretrained, few-shot ready |
| Compute | Fast | Requires GPU for large datasets |

---
## Section 2: Training the Regression Head

**Goals:**
- Train the frozen ESM-2 head (only MLP parameters updated)
- Train a LoRA-adapted ESM-2 (LoRA adapter + MLP parameters updated)
- Train a GP baseline with one-hot sequence encoding
- Compare Spearman correlation on the test set across all three models

### Architecture Overview

**Frozen ESM-2:**
```
ESM-2 encoder (frozen, 150M params) → mean pool → MLP head (trainable, ~330K params)
```

**LoRA ESM-2:**
```
ESM-2 encoder + LoRA adapters (trainable, ~1.2M params) → mean pool → MLP head (trainable)
```

LoRA (Low-Rank Adaptation) injects small trainable matrices into the query and value projection layers. The base ESM-2 weights remain frozen; only the adapters are updated.

In [ ]:
%%time
# Reduce epochs on CPU for speed
num_epochs = 50 if device == "cuda" else 10
print(f"Training for {num_epochs} epochs on {device}...")

frozen_model = ESM2DropoutModel(
    model_config=ESM2ModelConfig(),
    train_config=ESM2TrainConfig(num_epochs=num_epochs, log_frequency=num_epochs // 5),
    device=device,
)
frozen_model.train(train_data, val_data)

frozen_metrics = frozen_model.get_epoch_metrics()
print(f"Final train loss: {frozen_metrics[-1].train_loss:.4f}")
if frozen_metrics[-1].val_loss is not None:
    print(f"Final val loss:   {frozen_metrics[-1].val_loss:.4f}")

In [ ]:
assert len(frozen_metrics) == num_epochs, \
    f"Expected {num_epochs} epoch metrics, got {len(frozen_metrics)}"
assert frozen_metrics[-1].train_loss < frozen_metrics[0].train_loss, \
    "Train loss should decrease over training"
print("✓ Frozen model training epoch metrics verified")

In [ ]:
epochs     = [m.epoch for m in frozen_metrics]
train_loss = [m.train_loss for m in frozen_metrics]
val_loss   = [m.val_loss for m in frozen_metrics if m.val_loss is not None]
val_epochs = [m.epoch for m in frozen_metrics if m.val_loss is not None]
train_spearman = [m.additional_metrics.get("train_spearman", float("nan"))
                  for m in frozen_metrics]
val_spearman   = [m.additional_metrics.get("val_spearman", float("nan"))
                  for m in frozen_metrics if m.val_loss is not None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label="train loss", color="steelblue")
if val_loss:
    ax1.plot(val_epochs, val_loss, label="val loss", color="orange")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.set_title("Frozen ESM-2: Training Curves"); ax1.legend()

ax2.plot(epochs, train_spearman, label="train Spearman", color="steelblue")
if val_spearman:
    ax2.plot(val_epochs, val_spearman, label="val Spearman", color="orange")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Spearman ρ"); ax2.set_ylim(0, 1)
ax2.set_title("Frozen ESM-2: Spearman Correlation"); ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
frozen_preds = frozen_model.predict(list(test_data.candidates), with_uncertainty=False)
frozen_spearman = scipy.stats.spearmanr(
    frozen_preds.means, np.array(test_data.labels)
).statistic
print(f"Frozen ESM-2 test Spearman: {frozen_spearman:.3f}")
assert frozen_spearman > 0.3, f"Expected Spearman > 0.3, got {frozen_spearman:.3f}"
print("✓ Frozen model achieves reasonable Spearman correlation")

### LoRA Fine-Tuning

LoRA (Hu et al., 2022) injects trainable rank-16 matrices into the query and value projections of every ESM-2 attention layer. The base weights stay frozen; only the adapters (≈0.8% of total parameters) are updated. This lets the encoder adapt its representations to the GFP fitness landscape while retaining its evolutionary prior.

We implement LoRA as a standalone training pattern so every step is visible.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import EsmModel

def embed_sequences(encoder, tokenizer, sequences, device, batch_size=32):
    """Mean-pool ESM-2 hidden states over non-padding positions.

    NOTE: does NOT detach — call inside torch.no_grad() for inference,
    or leave gradients live for training with LoRA adapters.
    """
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]
        tokens = tokenizer(batch, return_tensors="pt", padding=True)
        input_ids = tokens["input_ids"].to(device)
        mask = tokens["attention_mask"].to(device)
        out = encoder(input_ids=input_ids, attention_mask=mask)
        m = mask.unsqueeze(-1).float()
        emb = (out.last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1e-9)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0)  # (N, 640)


def lora_mc_predict(encoder, head, tokenizer, candidates, device, num_samples=30):
    """MC Dropout predictions from a standalone LoRA model."""
    sequences = extract_sequences_from_inputs(candidates)
    encoder.eval()
    with torch.no_grad():
        x = embed_sequences(encoder, tokenizer, sequences, device).to(device)
    head.train()  # Keep dropout active for MC sampling
    samples = []
    with torch.no_grad():
        for _ in range(num_samples):
            samples.append(head(x).cpu().numpy())
    arr = np.stack(samples, axis=1)  # (N, num_samples)
    return arr.mean(axis=1), arr.var(axis=1), arr

In [ ]:
# Fresh ESM-2 encoder for LoRA (separate from frozen_model)
lora_enc = EsmModel.from_pretrained("facebook/esm2_t30_150M_UR50D").to(device)
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
)
lora_enc = get_peft_model(lora_enc, lora_config)
lora_enc.print_trainable_parameters()
# Expected: trainable params ~1.2M out of ~150M (≈0.8%)

# Regression head (same architecture as frozen model)
lora_head = ESM2RegressionHead(
    embedding_dim=640, hidden_dim=256, num_hidden_layers=2, dropout=0.1
).to(device)

# Reuse tokenizer from frozen_model
tokenizer = frozen_model.tokenizer

# Optimizer: head params + LoRA adapter params only
lora_adapter_params = [p for p in lora_enc.parameters() if p.requires_grad]
lora_optimizer = optim.Adam(
    list(lora_head.parameters()) + lora_adapter_params, lr=1e-3
)
criterion_lora = nn.MSELoss()

# Print trainable param counts
head_params  = sum(p.numel() for p in lora_head.parameters())
lora_params  = sum(p.numel() for p in lora_adapter_params)
total_params = sum(p.numel() for p in lora_enc.parameters())
print(f"MLP head params:   {head_params:,}")
print(f"LoRA adapter params: {lora_params:,}")
print(f"ESM-2 total params:  {total_params:,}")

In [ ]:
frozen_base_params = sum(
    p.numel() for p in lora_enc.parameters() if not p.requires_grad
)
assert frozen_base_params > 0, "Base ESM-2 weights should be frozen"
assert lora_params > 0, "LoRA adapter weights should be trainable"
assert lora_params < total_params * 0.02, \
    f"LoRA should be <2% of total params, got {lora_params/total_params:.1%}"
print(f"✓ LoRA adapters = {lora_params/total_params:.2%} of total parameters")

In [ ]:
%%time
num_epochs_lora = num_epochs  # Same as frozen for fair comparison
train_seqs = extract_sequences_from_inputs(train_data)
val_seqs   = extract_sequences_from_inputs(val_data)
train_labels_t = torch.tensor(train_data.labels, dtype=torch.float32).to(device)

lora_train_losses, lora_val_spearman_list = [], []
log_freq = max(1, num_epochs_lora // 5)

for epoch in range(num_epochs_lora):
    # --- Train step ---
    lora_enc.train()
    lora_head.train()
    lora_optimizer.zero_grad()                                    # zero before forward pass
    x_tr = embed_sequences(lora_enc, tokenizer, train_seqs, device).to(device)
    preds_tr = lora_head(x_tr).squeeze(-1)
    loss = criterion_lora(preds_tr, train_labels_t)
    loss.backward()
    lora_optimizer.step()
    lora_train_losses.append(loss.item())

    # --- Validation Spearman ---
    lora_enc.eval()
    lora_head.eval()
    with torch.no_grad():
        x_val = embed_sequences(lora_enc, tokenizer, val_seqs, device).to(device)
        preds_val = lora_head(x_val).squeeze(-1).cpu().numpy()
    spearman = scipy.stats.spearmanr(preds_val, val_data.labels).statistic
    lora_val_spearman_list.append(spearman)

    if epoch % log_freq == 0:
        print(f"Epoch {epoch:3d}/{num_epochs_lora} — "
              f"loss: {loss.item():.4f}, val Spearman: {spearman:.3f}")

print("LoRA training complete.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(lora_train_losses, label="LoRA train loss", color="darkorange")
ax1.plot(train_loss, label="Frozen train loss", color="steelblue", linestyle="--")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.set_title("Training Loss: Frozen vs LoRA ESM-2"); ax1.legend()

ax2.plot(lora_val_spearman_list, label="LoRA val Spearman", color="darkorange")
val_sp_full = [m.additional_metrics.get("val_spearman", float("nan"))
               for m in frozen_metrics]
ax2.plot(val_sp_full, label="Frozen val Spearman", color="steelblue", linestyle="--")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Spearman ρ"); ax2.set_ylim(0, 1)
ax2.set_title("Validation Spearman: Frozen vs LoRA ESM-2"); ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
lora_means, lora_vars, lora_samples = lora_mc_predict(
    lora_enc, lora_head, tokenizer,
    list(test_data.candidates), device, num_samples=30
)
lora_spearman = scipy.stats.spearmanr(lora_means, np.array(test_data.labels)).statistic
print(f"LoRA ESM-2 test Spearman: {lora_spearman:.3f}")

In [ ]:
%%time
print("Training GP baseline (Matérn-2.5 + one-hot encoding)...")
gp_model = GPModel(
    model_config=GPModelConfig(kernel_type="matern", matern_nu=2.5),
    featurizer_config=FeaturizerConfig(
        featurizer_type="one_hot",
        flatten_one_hot=True,
    ),
)
gp_model.train(train_data, val_data)
gp_preds = gp_model.predict(list(test_data.candidates), with_uncertainty=False)
gp_spearman = scipy.stats.spearmanr(
    gp_preds.means, np.array(test_data.labels)
).statistic
print(f"GP (Matérn-2.5, one-hot) test Spearman: {gp_spearman:.3f}")

In [ ]:
results = {
    "GP\n(Matérn-2.5,\none-hot)": gp_spearman,
    "Frozen\nESM-2": frozen_spearman,
    "LoRA\nESM-2": lora_spearman,
}
colours = ["#4CAF50", "#2196F3", "#FF9800"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(results.keys(), results.values(), color=colours, alpha=0.85,
              edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, 1)
ax.set_ylabel("Spearman ρ (test set)")
ax.set_title("GFP Fitness Prediction: Model Comparison\n(480 training sequences)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

### Key Takeaway — Section 2

| Model | Spearman ρ | Parameters trained | Best for |
|---|---|---|---|
| GP (Matérn-2.5, one-hot) | ~0.5–0.6 | kernel hyperparams | < 100 sequences |
| Frozen ESM-2 head | ~0.7–0.8 | ~330K (MLP only) | 100–1000 sequences, no GPU |
| LoRA ESM-2 | ~0.75–0.85 | ~1.5M (LoRA + MLP) | 1000+ sequences or GPU available |

LoRA improves over the frozen head by adapting the encoder's representations to the GFP fitness landscape. The GP baseline underperforms here because one-hot encoding ignores evolutionary context — on very small datasets (< 100 sequences), GP would be more competitive.